# 📡 FM Carrier Frequency Estimation — v3 (Audit-Corrected)
> **Formato CSV:** `id, mac, campaign_id, pxx (JSON array), start_freq_hz, end_freq_hz, timestamp_ms, lat, lng, …`  
> **SDR:** HackRF One · **Banda:** 87.5 – 108.0 MHz · **Resolución:** ~1220 Hz/bin (16 384 bins / 20 MHz)

### Correcciones aplicadas respecto a la versión anterior

| # | Problema | Severidad | Fix |
|---|---|---|---|
| FIX-1 | Desviación de reloj HackRF no calibrada | 10/10 | Estimación PPM desde canal de referencia + corrección afín al eje de frecuencias |
| FIX-2 | Pico DC / LO leakage detectado como portadora | 9/10 | Máscara de supresión ±`dc_mask_bins` en el bin central del sweep |
| FIX-3 | Fit Gaussiano en escala logarítmica (dBm) | 8/10 | Modo Gaussiano eliminado; solo interpolación cuadrática (correcta en dBm) |
| FIX-4 | Parábola convexa → offset hacia mínimo | 7/10 | Guarda de signo: si `c[0] ≥ 0` retorna offset=0 |
| FIX-5 | Colisión `datetime` módulo vs clase | 10/10 | `import datetime as dt` en todas las celdas |
| FIX-6 | `FrameworkConfig` / `DetectionConfig` duplicados; unidades mixtas | 8/10 | Clase única `SpectralConfig`, todas las frecuencias en Hz |
| FIX-7 | Exclusión por substring frágil (basename vs stem) | 9/10 | `Path(fpath).stem in exclude_stems` (set membership exacto) |
| FIX-8 | Promedio solo en lote, no en archivo individual | media | `load_node_psd()` siempre promedia; nunca se procesa una sola fila |
| FIX-9 | `interp_half_window=3` → ±3 662 Hz > tolerancia ±2 000 Hz | 7/10 | `interp_half_window=1` → ±1 221 Hz |

---
**Pipeline:**  
`CSV` → `parse_pxx_cell()` → `load_node_psd()` (avg lineal) → `apply_dc_mask()` → `apply_clock_correction()`  
→ `detect_carriers()` → `estimate_carrier()` → `run_batch_detection()` → `print_summary()`


## 0 · Entorno (Deepnote)

In [1]:
# ── Instala sólo lo que el entorno base de Deepnote no trae ─────────────────
import importlib, subprocess, sys

REQUIRED = {"plotly": "plotly>=5.18", "tabulate": "tabulate"}
for module, pkg in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed {pkg}")
    else:
        print(f"OK  {module}")


OK  plotly
OK  tabulate


## 1 · Importaciones
> **FIX-5:** `import datetime as dt` — evita la colisión entre el módulo y la clase `datetime.datetime`.

In [2]:
# FIX-5: alias obligatorio para evitar AttributeError en celdas posteriores
import datetime as dt

import warnings, glob, ast
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths
from tabulate import tabulate
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("Imports OK ✓")


Imports OK ✓


## 2 · Configuración unificada
> **FIX-6:** Una sola clase `SpectralConfig`; todas las frecuencias en Hz (sin mezcla MHz/Hz).  
> **FIX-9:** `interp_half_window = 1` → ±1 221 Hz, dentro del margen de tolerancia de ±2 000 Hz.  
> **FIX-3:** El campo `interpolation_method` ya no acepta `"gaussian"` — eliminado por error matemático en escala dBm.


In [3]:
@dataclass
class SpectralConfig:
    # ── Banda (FIX-6: límite inferior corregido a 87.5 MHz) ──────────────────
    fm_band_min_hz: float     = 87_500_000.0   # era 88.0 MHz → omitía 5 canales ITU-R
    fm_band_max_hz: float     = 108_000_000.0
    channel_raster_hz: float  = 100_000.0      # rejilla ITU-R BS.450-3

    # ── Hardware: corrección de reloj HackRF (FIX-1) ─────────────────────────
    apply_clock_correction: bool     = True
    # Canal de referencia para estimar el PPM. Elegir la estación más fuerte
    # y estable del área. None = usar clock_ppm_override directamente.
    clock_reference_hz: Optional[float] = 94_900_000.0   # ajustar según cobertura
    clock_ppm_override: float            = 0.0
    clock_ppm_estimated: float           = 0.0  # se rellena en tiempo de ejecución

    # ── Hardware: máscara de pico DC (FIX-2) ─────────────────────────────────
    # 16 384 bins / 20 MHz → 1 bin ≈ 1 220 Hz
    # Centro del sweep (88+108)/2 = 98 MHz → bin 8 192
    # ±5 bins ≈ ±6.1 kHz: suprime el spike sin afectar canales adyacentes
    dc_mask_bins: int = 5

    # ── Promediado (FIX-8) ────────────────────────────────────────────────────
    average_in_linear: bool = True   # promedio en lineal → dBm; NUNCA en dBm directo

    # ── Detección de picos ────────────────────────────────────────────────────
    min_prominence_db: float    = 5.0
    min_peak_distance_hz: float = 100_000.0

    # ── Interpolación sub-bin (FIX-3, FIX-4, FIX-9) ─────────────────────────
    interp_half_window: int = 1   # ±1 bin = ±1 221 Hz < tolerancia ±2 000 Hz
    # FIX-3: "gaussian" eliminado — ajuste Gaussiano en dBm es matemáticamente incorrecto.
    # Usar siempre interpolación cuadrática (parábola en dBm = log de Gaussiana en lineal).

    # ── Estimación de piso de ruido ───────────────────────────────────────────
    noise_guard_hz: float  = 200_000.0
    noise_window_hz: float = 500_000.0

    # ── Umbrales de confianza ─────────────────────────────────────────────────
    min_snr_full_db: float  = 20.0
    min_prom_full_db: float = 10.0

    # ── Cumplimiento (MINTIC Res. 415/2010) ───────────────────────────────────
    tolerance_hz: float = 2_000.0   # ±2 kHz límite duro
    warning_hz: float   = 1_000.0   # ±1 kHz límite suave


CFG = SpectralConfig()
print("SpectralConfig cargado ✓")
print(f"  Banda          : {CFG.fm_band_min_hz/1e6:.1f} – {CFG.fm_band_max_hz/1e6:.1f} MHz")
print(f"  Tolerancia     : ±{CFG.tolerance_hz/1e3:.1f} kHz")
print(f"  Ventana interp : ±{CFG.interp_half_window} bin(s) ≈ ±{CFG.interp_half_window*1220:.0f} Hz")
print(f"  DC mask        : ±{CFG.dc_mask_bins} bins ≈ ±{CFG.dc_mask_bins*1220:.0f} Hz")
print(f"  Ref. reloj     : {CFG.clock_reference_hz/1e6:.1f} MHz")


SpectralConfig cargado ✓
  Banda          : 87.5 – 108.0 MHz
  Tolerancia     : ±2.0 kHz
  Ventana interp : ±1 bin(s) ≈ ±1220 Hz
  DC mask        : ±5 bins ≈ ±6100 Hz
  Ref. reloj     : 94.9 MHz


## 3 · Parser PXX y cargador de nodo
> **FIX-8:** `load_node_psd()` siempre promedia todos los sweeps en escala lineal antes de convertir a dBm. Nunca se procesa una sola fila.

In [4]:
def parse_pxx_cell(cell: str) -> np.ndarray:
    """Convierte la columna pxx (string JSON) a array float64."""
    try:
        return np.array(ast.literal_eval(cell), dtype=np.float64)
    except Exception:
        return np.fromstring(str(cell).strip("[]"), sep=",", dtype=np.float64)


def load_node_psd(csv_path, cfg: SpectralConfig = CFG) -> Optional[dict]:
    """
    Carga un CSV de nodo, parsea TODAS las filas pxx y devuelve:
      node_name, freqs_hz, pxx (dBm promediado), n_sweeps,
      timestamp_first_ms, timestamp_last_ms, lat, lng

    FIX-8: el promedio en lineal se aplica siempre aquí, no condicionalmente.
    """
    path = Path(csv_path)
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        warnings.warn(f"No se pudo leer {path.name}: {exc}")
        return None

    if "pxx" not in df.columns or df.empty:
        return None

    arrays = []
    for cell in df["pxx"]:
        try:
            arr = parse_pxx_cell(str(cell))
            if arr.size > 0:
                arrays.append(arr)
        except Exception:
            continue

    if not arrays:
        return None

    # Alinear longitudes → moda (descartar filas corruptas)
    lengths  = np.array([a.size for a in arrays])
    mode_len = int(pd.Series(lengths).mode().iloc[0])
    arrays   = [a for a in arrays if a.size == mode_len]
    if not arrays:
        return None

    # FIX-8: promedio lineal obligatorio
    stack   = np.vstack(arrays)
    lin_avg = np.mean(10.0 ** (stack / 10.0), axis=0)
    pxx_avg = 10.0 * np.log10(np.maximum(lin_avg, 1e-30))

    meta = df.dropna(subset=["start_freq_hz", "end_freq_hz"])
    if meta.empty:
        warnings.warn(f"{path.name}: sin metadatos de frecuencia")
        return None

    row0     = meta.iloc[0]
    freqs_hz = np.linspace(float(row0["start_freq_hz"]),
                            float(row0["end_freq_hz"]),
                            mode_len, endpoint=True)

    ts      = df["timestamp"].dropna()
    lat_col = df["lat"].dropna() if "lat" in df.columns else pd.Series(dtype=float)
    lng_col = df["lng"].dropna() if "lng" in df.columns else pd.Series(dtype=float)

    return {
        "node_name":          path.stem,
        "freqs_hz":           freqs_hz,
        "pxx":                pxx_avg,
        "n_sweeps":           len(arrays),
        "timestamp_first_ms": int(ts.min()) if not ts.empty else None,
        "timestamp_last_ms":  int(ts.max()) if not ts.empty else None,
        "lat":  float(lat_col.iloc[0]) if not lat_col.empty else None,
        "lng":  float(lng_col.iloc[0]) if not lng_col.empty else None,
    }

print("Parser y cargador definidos ✓")


Parser y cargador definidos ✓


## 4 · Corrección de reloj HackRF One (FIX-1)

El TCXO del HackRF One tiene una desviación de fábrica de **±20 PPM**.  
A 108 MHz → **±2 160 Hz**, que consume el 100 % del margen de ±2 kHz de MINTIC.

**Estrategia:** encontrar el pico de mayor prominencia dentro de una ventana de ±50 kHz alrededor del canal de referencia, calcular el offset en PPM y aplicar una corrección afín a todo el eje de frecuencias:

```
f_corrected = f_measured × (1 − PPM / 1 000 000)
```

Si el canal de referencia no está en el sweep, se usa `clock_ppm_override` (defecto 0.0).


In [5]:
def _estimate_clock_ppm(freqs_hz: np.ndarray, pxx: np.ndarray,
                        cfg: SpectralConfig) -> float:
    """
    FIX-1: Estima el offset PPM del reloj del HackRF usando un canal de referencia.
    Devuelve el PPM estimado (positivo = HackRF mide MÁS alto de lo real).
    """
    if cfg.clock_reference_hz is None:
        return cfg.clock_ppm_override

    # Ventana de búsqueda: ±50 kHz alrededor del nominal de referencia
    search_hz = 50_000.0
    mask = np.abs(freqs_hz - cfg.clock_reference_hz) <= search_hz
    if mask.sum() < 3:
        warnings.warn(
            f"Canal de referencia {cfg.clock_reference_hz/1e6:.1f} MHz "
            "no encontrado en el sweep. Usando clock_ppm_override."
        )
        return cfg.clock_ppm_override

    p_sub = pxx[mask]
    f_sub = freqs_hz[mask]

    # Pico de máxima prominencia en la ventana
    peaks, props = find_peaks(p_sub, prominence=3.0)
    if len(peaks) == 0:
        # Sin pico claro: usar el argmax simple
        local_idx = int(np.argmax(p_sub))
    else:
        local_idx = peaks[int(np.argmax(props["prominences"]))]

    f_measured = float(f_sub[local_idx])
    ppm = (f_measured - cfg.clock_reference_hz) / cfg.clock_reference_hz * 1e6
    return float(ppm)


def apply_clock_correction(freqs_hz: np.ndarray, cfg: SpectralConfig) -> np.ndarray:
    """
    FIX-1: Aplica la corrección PPM estimada a todo el eje de frecuencias.
    f_corrected = f_measured * (1 - ppm / 1e6)
    """
    ppm = cfg.clock_ppm_estimated
    if abs(ppm) < 0.01:       # corrección despreciable
        return freqs_hz
    return freqs_hz * (1.0 - ppm / 1e6)


print("Corrección de reloj definida ✓")


Corrección de reloj definida ✓


## 5 · Máscara de pico DC / LO Leakage (FIX-2)

En arquitecturas de conversión directa (zero-IF) como el HackRF, el **bin central** del sweep contiene energía espuria del oscilador local.

Para 88–108 MHz (20 MHz de BW, 16 384 bins):  
- Bin central = 8 192 → frecuencia central = **98.0 MHz**  
- `dc_mask_bins = 5` → suprime 98.0 MHz ± 6.1 kHz

La supresión reemplaza los bins afectados con interpolación lineal de los bordes de la máscara (no con −∞, lo que rompería `find_peaks`).


In [6]:
def apply_dc_mask(freqs_hz: np.ndarray, pxx: np.ndarray,
                  cfg: SpectralConfig) -> np.ndarray:
    """
    FIX-2: Suprime el pico DC en el bin central del sweep.
    Reemplaza los bins enmascarados con interpolación lineal de los bordes.
    """
    n   = len(pxx)
    mid = n // 2
    lo  = max(0,     mid - cfg.dc_mask_bins)
    hi  = min(n - 1, mid + cfg.dc_mask_bins)

    pxx_clean = pxx.copy()

    # Interpolar linealmente entre los bordes de la máscara
    if lo > 0 and hi < n - 1:
        pxx_clean[lo:hi + 1] = np.linspace(pxx[lo - 1], pxx[hi + 1], hi - lo + 1)
    else:
        # En el borde del array: rellenar con la mediana global
        pxx_clean[lo:hi + 1] = np.median(pxx)

    masked_freq_mhz = freqs_hz[mid] / 1e6
    return pxx_clean


print("Máscara DC definida ✓")
print(f"  Bin central (16384 bins, 88-108 MHz): bin 8192 → 98.000 MHz")
print(f"  Con dc_mask_bins={CFG.dc_mask_bins}: suprime 98.000 MHz ± {CFG.dc_mask_bins*1220/1e3:.1f} kHz")


Máscara DC definida ✓
  Bin central (16384 bins, 88-108 MHz): bin 8192 → 98.000 MHz
  Con dc_mask_bins=5: suprime 98.000 MHz ± 6.1 kHz


## 6 · Helpers espectrales

### FIX-3 + FIX-4: Interpolación cuadrática corregida

**FIX-3:** El ajuste Gaussiano estándar `A·exp(…)` aplicado sobre datos en dBm es matemáticamente incorrecto: la envolvente de una portadora es Gaussiana en escala **lineal**, que en dBm es una **parábola** (log de Gaussiana). Modo Gaussiano eliminado.

**FIX-4:** Si `polyfit` devuelve `c[0] ≥ 0`, la parábola abre hacia arriba (convexa) y su vértice es un **mínimo**, no un máximo. La corrección es devolver `offset = 0.0` en ese caso.


In [7]:
def _quadratic_offset(p: np.ndarray, idx: int, hw: int) -> float:
    """
    FIX-3: Solo interpolación cuadrática (correcta para datos en dBm).
    FIX-4: Guarda de concavidad — si c[0] >= 0 la parábola es convexa → offset 0.
    FIX-9: hw=1 por defecto en SpectralConfig → ±1 bin ≈ ±1221 Hz.
    """
    lo, hi = max(0, idx - hw), min(len(p) - 1, idx + hw)
    if hi - lo < 2:          # no hay suficientes puntos para el ajuste
        return 0.0
    x = np.arange(lo, hi + 1, dtype=float) - idx
    try:
        c = np.polyfit(x, p[lo:hi + 1], 2)
    except np.linalg.LinAlgError:
        return 0.0

    # FIX-4: c[0] < 0 → cóncava → vértice es máximo (correcto)
    #        c[0] >= 0 → convexa → vértice es mínimo (incorrecto, descartar)
    if c[0] >= 0:
        return 0.0

    offset = -c[1] / (2.0 * c[0])
    return float(np.clip(offset, -hw, hw))


def _noise_floor(p: np.ndarray, idx: int, df_hz: float,
                 cfg: SpectralConfig) -> float:
    """Mediana del PSD en ventanas laterales al pico (fuera de la banda de guarda)."""
    gb = max(1, int(cfg.noise_guard_hz / df_hz))
    nw = max(1, int(cfg.noise_window_hz / df_hz))
    n  = len(p)
    samples = np.concatenate([
        p[max(0, idx - gb - nw) : max(0, idx - gb)],
        p[min(n, idx + gb)      : min(n, idx + gb + nw)],
    ])
    return float(np.median(samples)) if samples.size > 0 else float(np.median(p))


def _prominence(p: np.ndarray, idx: int, df_hz: float,
                cfg: SpectralConfig) -> float:
    """Prominencia del pico; fallback robusto si find_peaks no lo incluye."""
    peaks_all, props = find_peaks(p, prominence=0)
    where = np.where(peaks_all == idx)[0]
    if where.size:
        return float(props["prominences"][where[0]])
    hd = max(1, int(cfg.min_peak_distance_hz / df_hz))
    lo, hi = max(0, idx - hd), min(len(p) - 1, idx + hd)
    base = min(
        p[lo:idx].min()      if idx > lo else p[idx],
        p[idx:hi + 1].min()  if hi > idx else p[idx],
    )
    return float(p[idx] - base)


def _confidence(snr_db: float, prom_db: float, width_bins: float,
                cfg: SpectralConfig):
    flags  = []
    s_snr  = min(1.0, max(0.0, snr_db  / cfg.min_snr_full_db))
    s_prom = min(1.0, max(0.0, prom_db / cfg.min_prom_full_db))
    if s_snr  < 0.5: flags.append(f"LOW_SNR({snr_db:.1f}dB)")
    if s_prom < 0.5: flags.append(f"LOW_PROM({prom_db:.1f}dB)")
    s_sharp = max(0.0, 1.0 - max(0.0, width_bins - 5) / 10) if width_bins > 5 else 1.0
    if s_sharp < 1.0: flags.append(f"WIDE({width_bins:.1f}bins)")
    return round(s_snr * 0.45 + s_prom * 0.40 + s_sharp * 0.15, 4), flags


def _nearest_channel(est_hz: float, cfg: SpectralConfig) -> Optional[float]:
    """Retorna el canal ITU-R más cercano si está dentro de 5× la tolerancia."""
    ch  = round((est_hz - cfg.fm_band_min_hz) / cfg.channel_raster_hz)
    nom = cfg.fm_band_min_hz + ch * cfg.channel_raster_hz
    return round(nom / 1e6, 1) if abs(est_hz - nom) <= cfg.tolerance_hz * 5 else None


print("Helpers espectrales definidos ✓")


Helpers espectrales definidos ✓


## 7 · Estimación de una portadora

In [8]:
def _estimate_one(node_name: str, freqs_hz: np.ndarray, p: np.ndarray,
                  peak_idx: int, cfg: SpectralConfig) -> dict:
    """Estima todos los measurands para un único pico detectado."""
    df_hz    = float(np.median(np.diff(freqs_hz)))
    offset   = _quadratic_offset(p, peak_idx, cfg.interp_half_window)
    est_hz   = freqs_hz[peak_idx] + offset * df_hz
    est_mhz  = est_hz / 1e6

    try:
        w_bins = float(peak_widths(p, [peak_idx], rel_height=0.5)[0][0])
    except Exception:
        w_bins = 0.0

    prom_db  = _prominence(p, peak_idx, df_hz, cfg)
    nf_dbm   = _noise_floor(p, peak_idx, df_hz, cfg)
    snr_db   = float(p[peak_idx] - nf_dbm)
    conf, flags = _confidence(snr_db, prom_db, w_bins, cfg)

    nom_mhz  = _nearest_channel(est_hz, cfg)
    dev_hz   = (est_mhz - nom_mhz) * 1e6 if nom_mhz else None
    dev_khz  = dev_hz / 1e3               if dev_hz is not None else None
    dev_ppm  = dev_hz / (nom_mhz * 1e6) * 1e6 if (dev_hz is not None and nom_mhz) else None

    if   dev_hz is None:                       status = "UNVERIFIABLE"
    elif abs(dev_hz) <= cfg.warning_hz:        status = "COMPLIANT"
    elif abs(dev_hz) <= cfg.tolerance_hz:      status = "WARNING"
    else:                                      status = "VIOLATION"

    return {
        "node":                node_name,
        "nominal_mhz":         nom_mhz,
        "estimated_mhz":       round(est_mhz, 6),
        "peak_bin_mhz":        round(float(freqs_hz[peak_idx]) / 1e6, 6),
        "deviation_hz":        round(dev_hz,  2) if dev_hz  is not None else None,
        "deviation_khz":       round(dev_khz, 4) if dev_khz is not None else None,
        "deviation_ppm":       round(dev_ppm, 4) if dev_ppm is not None else None,
        "spike_prominence_db": round(prom_db,  2),
        "spike_width_hz":      round(w_bins * df_hz, 1),
        "spike_width_bins":    round(w_bins,   2),
        "peak_power_dbm":      round(float(p[peak_idx]), 2),
        "snr_db":              round(snr_db,   2),
        "noise_floor_dbm":     round(nf_dbm,   2),
        "confidence":          conf,
        "confidence_flags":    "|".join(flags) if flags else "",
        "compliance_status":   status,
    }

print("Estimador de portadora definido ✓")


Estimador de portadora definido ✓


## 8 · Batch runner

In [9]:
def run_batch_detection(pxx_node: dict, cfg: SpectralConfig = CFG,
                        verbose: bool = True) -> pd.DataFrame:
    """
    Recorre todos los nodos en pxx_node, aplica FIX-1 (clock) y FIX-2 (DC mask),
    detecta portadoras y estima measurands.

    Entrada: {node_name: {"freqs_hz", "pxx", "n_sweeps", ...}}
    Salida : DataFrame — una fila por portadora detectada.
    """
    records, skipped = [], []

    for node_name in sorted(pxx_node.keys()):
        entry  = pxx_node[node_name]
        freqs  = np.asarray(entry["freqs_hz"], dtype=np.float64)
        p      = np.asarray(entry["pxx"],      dtype=np.float64)
        n_sw   = entry.get("n_sweeps", "?")
        label  = node_name.split("/")[-1]

        if p.size < 10:
            skipped.append(node_name)
            continue

        # FIX-1: estimar PPM desde este nodo y aplicar corrección al eje de freq
        if cfg.apply_clock_correction:
            ppm = _estimate_clock_ppm(freqs, p, cfg)
            cfg.clock_ppm_estimated = ppm
            freqs = apply_clock_correction(freqs, cfg)
        else:
            ppm = 0.0

        # FIX-2: suprimir pico DC central
        p_clean = apply_dc_mask(freqs, p, cfg)

        # Detección de picos
        df_hz     = float(np.median(np.diff(freqs)))
        dist_bins = max(1, int(cfg.min_peak_distance_hz / df_hz))
        peaks, _  = find_peaks(p_clean, prominence=cfg.min_prominence_db,
                                distance=dist_bins)

        if verbose:
            print(f"  {label:<28}  sweeps={str(n_sw):>3}  "
                  f"carriers={len(peaks):>2}  PPM={ppm:+.2f}")

        for idx in peaks:
            records.append(_estimate_one(node_name, freqs, p_clean, int(idx), cfg))

    if skipped and verbose:
        print(f"\n  Skipped (vacíos): {[s.split('/')[-1] for s in skipped]}")

    if not records:
        print("Sin portadoras detectadas.")
        return pd.DataFrame()

    col_order = [
        "node", "nominal_mhz", "estimated_mhz", "peak_bin_mhz",
        "deviation_hz", "deviation_khz", "deviation_ppm",
        "spike_prominence_db", "spike_width_hz", "spike_width_bins",
        "peak_power_dbm", "snr_db", "noise_floor_dbm",
        "confidence", "confidence_flags", "compliance_status",
    ]
    df = pd.DataFrame(records)
    return df[[c for c in col_order if c in df.columns]]

print("Batch runner definido ✓")


Batch runner definido ✓


## 9 · Cargador por glob y tablas de resumen

In [10]:
def build_pxx_node_averaged(pattern: str, exclude_stems: set = None,
                            cfg: SpectralConfig = CFG,
                            verbose: bool = True) -> dict:
    """
    FIX-7: exclusión por Path(fpath).stem — comparación exacta de conjunto.
           Antes se usaba substring frágil que podía excluir nodos no deseados.
    FIX-8: load_node_psd() ya garantiza el promedio lineal internamente.
    """
    exclude_stems = exclude_stems or set()
    result, errors = {}, []

    for fpath in sorted(glob.glob(pattern, recursive=True)):
        stem = Path(fpath).stem              # FIX-7: solo nombre de archivo, sin ruta
        if stem in exclude_stems:            # FIX-7: set membership exacto
            if verbose:
                print(f"  Excluded  {stem}")
            continue

        entry = load_node_psd(fpath, cfg)
        if entry is None:
            errors.append(fpath)
            continue

        result[entry["node_name"]] = entry
        if verbose:
            ts = entry["timestamp_first_ms"]
            ts_str = (dt.datetime.utcfromtimestamp(ts / 1e3).strftime("%Y-%m-%d %H:%M")
                      if ts else "unknown")
            print(f"  Loaded    {entry['node_name']:<28}  "
                  f"sweeps={entry['n_sweeps']:>3}  bins={entry['pxx'].size:>6}  t0={ts_str}")

    if errors and verbose:
        print(f"\n  Fallos ({len(errors)}): {[Path(e).name for e in errors]}")
    return result


# ── Tablas de resumen ─────────────────────────────────────────────────────────

def carrier_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    status_rank = {"VIOLATION": 3, "WARNING": 2, "COMPLIANT": 1, "UNVERIFIABLE": 0}
    out = df.groupby("nominal_mhz", dropna=False).agg(
        n_nodes         =("node",                "nunique"),
        dev_hz_mean     =("deviation_hz",        "mean"),
        dev_hz_std      =("deviation_hz",        "std"),
        dev_hz_max_abs  =("deviation_hz",        lambda x: x.abs().max()),
        snr_mean_db     =("snr_db",              "mean"),
        snr_min_db      =("snr_db",              "min"),
        prominence_mean =("spike_prominence_db", "mean"),
        confidence_mean =("confidence",          "mean"),
        confidence_min  =("confidence",          "min"),
    ).reset_index()
    worst = (
        df.assign(_r=df["compliance_status"].map(status_rank))
          .groupby("nominal_mhz", dropna=False)
          .apply(lambda g: g.loc[g["_r"].idxmax(), "compliance_status"])
          .reset_index(name="worst_status")
    )
    return out.merge(worst, on="nominal_mhz", how="left").sort_values("nominal_mhz").round(3)


def node_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df
    counts = df.groupby(["node", "compliance_status"]).size().unstack(fill_value=0)
    for col in ["COMPLIANT", "WARNING", "VIOLATION", "UNVERIFIABLE"]:
        if col not in counts.columns: counts[col] = 0
    agg = df.groupby("node").agg(
        n_carriers      =("estimated_mhz",   "count"),
        snr_mean_db     =("snr_db",           "mean"),
        snr_min_db      =("snr_db",           "min"),
        confidence_mean =("confidence",       "mean"),
        confidence_min  =("confidence",       "min"),
        dev_hz_mean_abs =("deviation_hz",     lambda x: x.abs().mean()),
        dev_hz_max_abs  =("deviation_hz",     lambda x: x.abs().max()),
    ).reset_index().round(3)
    out = agg.merge(
        counts[["COMPLIANT", "WARNING", "VIOLATION", "UNVERIFIABLE"]].reset_index(),
        on="node", how="left",
    )
    out["node_label"] = out["node"].str.split("/").str[-1]
    cols = ["node_label", "n_carriers", "COMPLIANT", "WARNING", "VIOLATION",
            "UNVERIFIABLE", "snr_mean_db", "snr_min_db", "confidence_mean",
            "confidence_min", "dev_hz_mean_abs", "dev_hz_max_abs", "node"]
    return out[[c for c in cols if c in out.columns]].sort_values("node_label")


def print_summary(df: pd.DataFrame, cfg: SpectralConfig = CFG) -> None:
    sep = "═" * 96
    if df.empty:
        print(f"\n{sep}\n  Sin portadoras detectadas.\n{sep}\n")
        return

    ppm = cfg.clock_ppm_estimated
    print(f"\n{sep}")
    print(f"  FM CARRIER ESTIMATION — BATCH SUMMARY  (v3 — audit-corrected)")
    print(f"  Nodos: {df['node'].nunique()}  |  Detecciones: {len(df)}  |  "
          f"Tolerancia: ±{cfg.tolerance_hz/1e3:.1f} kHz  |  "
          f"PPM aplicado: {ppm:+.2f}  |  "
          f"Máscara DC: ±{cfg.dc_mask_bins} bins  |  "
          f"Ventana interp: ±{cfg.interp_half_window} bin(s)")
    print(sep)

    vc       = df["compliance_status"].value_counts()
    icon_map = {"COMPLIANT": "✅", "WARNING": "⚠️ ", "VIOLATION": "🚨", "UNVERIFIABLE": "❓"}
    print(f"\n  CUMPLIMIENTO")
    for s in ["COMPLIANT", "WARNING", "VIOLATION", "UNVERIFIABLE"]:
        n = vc.get(s, 0)
        print(f"    {icon_map[s]} {s:<14} {n:>3}  {'█' * n}")

    cs = carrier_summary(df)
    if not cs.empty:
        rows = []
        for _, r in cs.iterrows():
            rows.append([
                f"{r['nominal_mhz']:.1f}" if pd.notna(r["nominal_mhz"]) else "N/A",
                int(r["n_nodes"]),
                f"{r['dev_hz_mean']:+.1f}" if pd.notna(r["dev_hz_mean"]) else "N/A",
                f"{r['dev_hz_std']:.1f}"   if pd.notna(r["dev_hz_std"])  else "—",
                f"{r['dev_hz_max_abs']:.1f}" if pd.notna(r["dev_hz_max_abs"]) else "N/A",
                f"{r['snr_mean_db']:.1f}",
                f"{r['confidence_mean']:.3f}",
                f"{icon_map.get(r.get('worst_status',''),'')}{r.get('worst_status','')}",
            ])
        print(f"\n  POR CANAL (cross-nodo)")
        print(tabulate(rows,
            headers=["Canal(MHz)", "#Nodos", "Dev̄(Hz)", "σ(Hz)", "|D|max(Hz)",
                     "SNR̄(dB)", "Conf̄", "Peor estado"],
            tablefmt="github"))

    ns = node_summary(df)
    if not ns.empty:
        rows2 = []
        for _, r in ns.iterrows():
            rows2.append([
                r["node_label"],
                int(r["n_carriers"]),
                int(r["COMPLIANT"]), int(r["WARNING"]), int(r["VIOLATION"]),
                f"{r['snr_mean_db']:.1f}", f"{r['confidence_min']:.3f}",
                f"{r['dev_hz_mean_abs']:.1f}", f"{r['dev_hz_max_abs']:.1f}",
            ])
        print(f"\n  POR NODO")
        print(tabulate(rows2,
            headers=["Nodo", "Cars", "OK", "WARN", "VIOL",
                     "SNR̄", "Conf_min", "|D|̄Hz", "|D|max"],
            tablefmt="github"))

    print(f"\n{sep}\n")

print("Cargador, resúmenes y print_summary definidos ✓")


Cargador, resúmenes y print_summary definidos ✓


## 10 · Ejecución

Ajusta `ROOT_PATTERN` al path donde tengas los CSVs en Deepnote:

| Escenario | Pattern |
|---|---|
| Archivos en raíz del proyecto | `/work/Node*.csv` |
| En subcarpeta | `/work/DatasetFM-no-gains-88M-108M/Node*.csv` |
| Dataset largo | `/work/DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node*.csv` |


In [11]:
# ── Rutas — ajustar según estructura de /work en Deepnote ────────────────────
ROOT_PATTERN = "/work/DatasetFM-no-gains-88M-108M/Node*.csv"

# FIX-7: nombres exactos de stem (sin ruta, sin extensión)
EXCLUDE_STEMS = {
    "Node8-Bogota",     # datos insuficientes
    # "Node5-Bogota",   # atípico
    # "Node6-Bogota",   # mux roto
    # "Node3-Bogota",   # sin antena
}

# ── 1. Cargar y promediar ─────────────────────────────────────────────────────
print("Cargando CSVs y promediando sweeps...")
pxx_node = build_pxx_node_averaged(ROOT_PATTERN, EXCLUDE_STEMS, CFG, verbose=True)
print(f"\nNodos cargados: {len(pxx_node)}\n")

# ── 2. Detectar y estimar ─────────────────────────────────────────────────────
print("Ejecutando detección por lotes...")
summary_df = run_batch_detection(pxx_node, cfg=CFG, verbose=True)

# ── 3. Reporte ────────────────────────────────────────────────────────────────
print_summary(summary_df, cfg=CFG)

# ── 4. DataFrames derivados ───────────────────────────────────────────────────
carrier_df  = carrier_summary(summary_df)
node_df     = node_summary(summary_df)
violations  = summary_df[summary_df["compliance_status"] == "VIOLATION"]
warnings_df = summary_df[summary_df["compliance_status"] == "WARNING"]
low_conf    = summary_df[summary_df["confidence"] < 0.5]

print("DataFrames listos:")
print(f"  summary_df   → {len(summary_df):>4} filas  (una por portadora detectada)")
print(f"  carrier_df   → {len(carrier_df):>4} filas  (estadísticas cross-nodo por canal)")
print(f"  node_df      → {len(node_df):>4} filas  (agregado por nodo)")
print(f"  violations   → {len(violations):>4} portadoras")
print(f"  warnings_df  → {len(warnings_df):>4} portadoras")
print(f"  low_conf     → {len(low_conf):>4} portadoras  (confidence < 0.5)")

# Guard: skip derived frames if summary is empty
if summary_df.empty:
    carrier_df = pd.DataFrame()
    node_df    = pd.DataFrame()
    violations = warnings_df = low_conf = pd.DataFrame()


Cargando CSVs y promediando sweeps...
  Loaded    Node1-Bogota                  sweeps= 14  bins= 16384  t0=2026-03-09 16:42
  Loaded    Node2-Bogota                  sweeps= 14  bins= 16384  t0=2026-03-09 16:42
  Loaded    Node3-Bogota                  sweeps= 14  bins= 16384  t0=2026-03-09 16:42
  Loaded    Node5-Bogota                  sweeps= 14  bins= 16384  t0=2026-03-09 16:42

Nodos cargados: 4

Ejecutando detección por lotes...
  Node1-Bogota                  sweeps= 14  carriers=22  PPM=+11.13
  Node2-Bogota                  sweeps= 14  carriers=25  PPM=+11.13
  Node3-Bogota                  sweeps= 14  carriers=29  PPM=-14.60
  Node5-Bogota                  sweeps= 14  carriers=22  PPM=+23.99

════════════════════════════════════════════════════════════════════════════════════════════════
  FM CARRIER ESTIMATION — BATCH SUMMARY  (v3 — audit-corrected)
  Nodos: 4  |  Detecciones: 98  |  Tolerancia: ±2.0 kHz  |  PPM aplicado: +23.99  |  Máscara DC: ±5 bins  |  Ventana interp: ±

## 11 · Visualización rápida

In [12]:
if not summary_df.empty:
    COLOR_MAP = {"COMPLIANT": "#2ecc71", "WARNING": "#f39c12",
                 "VIOLATION": "#e74c3c", "UNVERIFIABLE": "#95a5a6"}

    # ── 11.1 Desviación por canal ─────────────────────────────────────────────
    fig = go.Figure()
    for status, grp in summary_df.dropna(subset=["deviation_hz"]).groupby("compliance_status"):
        fig.add_trace(go.Bar(
            x=grp["nominal_mhz"].astype(str),
            y=grp["deviation_hz"],
            name=status,
            marker_color=COLOR_MAP.get(status, "gray"),
        ))
    fig.add_hline(y= CFG.tolerance_hz, line=dict(color="red",  dash="dash", width=1),
                  annotation_text=f"+{CFG.tolerance_hz/1e3:.0f} kHz límite")
    fig.add_hline(y=-CFG.tolerance_hz, line=dict(color="red",  dash="dash", width=1))
    fig.add_hline(y= CFG.warning_hz,   line=dict(color="orange",dash="dot",  width=1))
    fig.add_hline(y=-CFG.warning_hz,   line=dict(color="orange",dash="dot",  width=1))
    fig.update_layout(
        title="Desviación de Frecuencia por Canal FM",
        xaxis_title="Canal nominal (MHz)",
        yaxis_title="Desviación (Hz)",
        template="plotly_dark", height=400,
        barmode="group", legend=dict(orientation="h", y=1.08),
    )
    fig.show()

    # ── 11.2 Confianza vs SNR ─────────────────────────────────────────────────
    fig2 = go.Figure()
    for status, grp in summary_df.groupby("compliance_status"):
        fig2.add_trace(go.Scatter(
            x=grp["snr_db"], y=grp["confidence"],
            mode="markers",
            name=status,
            marker=dict(color=COLOR_MAP.get(status, "gray"), size=9, opacity=0.8),
            text=grp["nominal_mhz"].astype(str) + " MHz — " + grp["node"],
            hovertemplate="%{text}<br>SNR: %{x:.1f} dB<br>Conf: %{y:.3f}<extra></extra>",
        ))
    fig2.update_layout(
        title="Confianza vs SNR por Portadora",
        xaxis_title="SNR (dB)",
        yaxis_title="Confidence [0–1]",
        template="plotly_dark", height=380,
    )
    fig2.show()
else:
    print("summary_df vacío — sin gráficas.")


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c71e238e-fa4e-44ac-af7f-d29021a4ac02' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>